# NSE 5× Turnaround V5 — Causal Research & Tradability Notebook

**Data constraint:** this notebook uses only the user's existing stock OHLCV Parquet files and the NIFTY50 OHLC Parquet file.

**No external fundamentals, FII/DII, delivery, sector, market-cap, API, or auxiliary datasets are used.**

### Core execution rule
- Signal/features are known at **stock market close T**
- Entry/exit is executed at **Open T+1**
- No T+1 High/Low/Close is used to decide the T+1 trade
- Stops are modeled as next-session execution under the close-to-next-open framework

### Main research questions
1. Does cross-sectional ranking improve the V4-style strategy?
2. Does NIFTY regime scaling improve robustness?
3. Do ATR-based position sizing and exits improve risk-adjusted returns?
4. Does the result survive realistic costs, slippage, liquidity constraints, walk-forward testing, and Monte Carlo?

## 1. Environment & configuration

The notebook is designed for Google Colab with Google Drive mounted.

Default paths:
- Stocks: `/content/drive/MyDrive/quant/data/parquet`
- NIFTY50: `/content/drive/MyDrive/quant/data/indices/nifty50/NIFTY50.parquet`
- Results: `/content/drive/MyDrive/quant/data/results/5x_turnaround_v5`

In [ ]:
from pathlib import Path
import json, math, warnings, os, sys, subprocess, importlib.util
warnings.filterwarnings("ignore")

DATA_DIR = Path("/content/drive/MyDrive/quant/data/parquet")
NIFTY_PATH = Path("/content/drive/MyDrive/quant/data/indices/nifty50/NIFTY50.parquet")
RESULTS_DIR = Path("/content/drive/MyDrive/quant/data/results/5x_turnaround_v5")

STOCK_FEATURES_PATH = RESULTS_DIR / "features" / "stock_features.parquet"
NIFTY_REGIME_PATH = RESULTS_DIR / "nifty" / "nifty_regime.parquet"
TRADES_PATH = RESULTS_DIR / "trades" / "trades.parquet"
EQUITY_PATH = RESULTS_DIR / "portfolio" / "equity_curve.parquet"
WF_RESULTS_PATH = RESULTS_DIR / "walk_forward" / "walk_forward_results.csv"
MC_RESULTS_PATH = RESULTS_DIR / "monte_carlo" / "monte_carlo_results.csv"

for p in [
    RESULTS_DIR / "features", RESULTS_DIR / "nifty", RESULTS_DIR / "trades",
    RESULTS_DIR / "portfolio", RESULTS_DIR / "walk_forward", RESULTS_DIR / "monte_carlo",
    RESULTS_DIR / "reports"
]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("NIFTY_PATH:", NIFTY_PATH)
print("RESULTS_DIR:", RESULTS_DIR)

In [ ]:
# Colab-only Drive mount; harmless if already mounted.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print("Drive mount skipped:", e)

required = ["pandas", "numpy", "pyarrow", "duckdb", "matplotlib", "scipy"]
missing = [x for x in required if importlib.util.find_spec(x) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

In [ ]:
import pandas as pd
import numpy as np
import pyarrow
import duckdb
import matplotlib.pyplot as plt
from scipy.stats import rankdata

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("duckdb:", duckdb.__version__)

## 2. Strategy configuration

All research parameters are centralized here. Do not tune them on the final out-of-sample period.

In [ ]:
CFG = {
    # Portfolio
    "initial_capital": 1_000_000.0,
    "max_positions": 10,
    "max_position_weight": 0.10,
    "max_total_exposure": 1.00,

    # Ranking
    "weights": {
        "momentum": 0.25,
        "relative_strength": 0.25,
        "trend": 0.20,
        "breakout": 0.15,
        "volume": 0.10,
        "volatility": 0.05,
    },

    # Entry
    "min_price": 20.0,
    "min_history": 252,
    "entry_top_pct": 0.10,
    "require_breakout": True,

    # NIFTY regime exposure
    "regime_exposure": {
        "BULL": 1.00,
        "MIXED": 0.50,
        "BEAR": 0.00,
        "UNKNOWN": 0.00,
    },

    # Position sizing
    "risk_budget_per_position": 0.01,
    "atr_stop_multiple": 5.0,
    "position_sizing": "volatility",

    # Exit
    "exit_mode": "ATR_TRAIL",
    "atr_trail_multiple": 5.0,
    "time_stop_days": 90,
    "momentum_exit_percentile": 0.20,

    # Costs / execution
    "slippage_bps": 20.0,
    "brokerage_bps": 0.0,
    "stt_bps": 10.0,
    "exchange_bps": 3.25,
    "gst_rate": 0.18,
    "stamp_bps": 1.5,
    "sebi_bps": 0.0001,

    # Liquidity
    "max_participation_pct": 0.05,

    # Research
    "mc_simulations": 10000,
    "random_seed": 42,
}

print(json.dumps(CFG, indent=2))

## 3. Helper functions

In [ ]:
def true_range(df):
    prev_close = df.groupby("Symbol")["Close"].shift(1)
    return pd.concat([
        df["High"] - df["Low"],
        (df["High"] - prev_close).abs(),
        (df["Low"] - prev_close).abs()
    ], axis=1).max(axis=1)

def rolling_return(s, n):
    return s / s.shift(n) - 1.0

def safe_pct_rank(s):
    return s.rank(pct=True, method="average")

def annualized_return(equity, periods_per_year=252):
    if len(equity) < 2:
        return np.nan
    years = (equity.index[-1] - equity.index[0]).days / 365.25
    if years <= 0 or equity.iloc[0] <= 0:
        return np.nan
    return (equity.iloc[-1] / equity.iloc[0]) ** (1 / years) - 1

def max_drawdown(equity):
    peak = equity.cummax()
    dd = equity / peak - 1
    return dd.min()

def performance_metrics(equity_df):
    x = equity_df.copy()
    x["Date"] = pd.to_datetime(x["Date"])
    x = x.sort_values("Date").drop_duplicates("Date").set_index("Date")
    eq = x["PortfolioValue"].astype(float)
    ret = eq.pct_change().fillna(0)
    cagr = annualized_return(eq)
    vol = ret.std(ddof=1) * np.sqrt(252) if len(ret) > 1 else np.nan
    sharpe = (ret.mean() / ret.std(ddof=1) * np.sqrt(252)) if ret.std(ddof=1) > 0 else np.nan
    downside = ret.where(ret < 0, 0)
    downside_dev = downside.std(ddof=1) * np.sqrt(252) if len(ret) > 1 else np.nan
    sortino = (ret.mean() * 252 / downside_dev) if downside_dev and downside_dev > 0 else np.nan
    dd = max_drawdown(eq)
    calmar = cagr / abs(dd) if pd.notna(cagr) and dd < 0 else np.nan
    return {
        "Start": x.index.min(),
        "End": x.index.max(),
        "InitialCapital": eq.iloc[0],
        "FinalCapital": eq.iloc[-1],
        "TotalReturn": eq.iloc[-1] / eq.iloc[0] - 1,
        "CAGR": cagr,
        "Volatility": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "MaxDrawdown": dd,
        "Calmar": calmar,
        "Days": len(x),
    }

def trade_metrics(trades):
    if trades.empty:
        return {}
    r = trades["NetReturn"].astype(float)
    wins = r[r > 0]
    losses = r[r < 0]
    pf = wins.sum() / abs(losses.sum()) if len(losses) else np.inf
    return {
        "Trades": len(trades),
        "WinRate": (r > 0).mean(),
        "AvgTrade": r.mean(),
        "MedianTrade": r.median(),
        "AvgWinner": wins.mean() if len(wins) else np.nan,
        "AvgLoser": losses.mean() if len(losses) else np.nan,
        "ProfitFactor": pf,
        "BestTrade": r.max(),
        "WorstTrade": r.min(),
        "AvgHoldingDays": trades["HoldingDays"].mean(),
        "MedianHoldingDays": trades["HoldingDays"].median(),
        "Turnover": trades["Turnover"].sum(),
    }

def print_metrics(metrics):
    for k, v in metrics.items():
        if isinstance(v, (float, np.floating)):
            print(f"{k:22s}: {v:.4f}")
        else:
            print(f"{k:22s}: {v}")

## 4. Discover and load stock Parquets

In [ ]:
stock_files = sorted(DATA_DIR.rglob("*.parquet"))
print("Stock parquet files:", len(stock_files))
print("First files:", [p.name for p in stock_files[:10]])
assert stock_files, f"No Parquet files found under {DATA_DIR}"
assert NIFTY_PATH.exists(), f"NIFTY file not found: {NIFTY_PATH}"

In [ ]:
def normalize_stock(df, source_name):
    cols = {c.lower().strip(): c for c in df.columns}
    aliases = {
        "date": ["date", "datetime", "timestamp"],
        "symbol": ["symbol", "ticker", "security", "stock"],
        "open": ["open"],
        "high": ["high"],
        "low": ["low"],
        "close": ["close", "adj close", "adj_close"],
        "volume": ["volume", "vol", "qty", "quantity", "total traded quantity"],
    }
    out = {}
    for target, names in aliases.items():
        found = next((cols[n] for n in names if n in cols), None)
        if found is not None:
            out[target] = found
    required = ["date", "open", "high", "low", "close", "volume"]
    missing = [x for x in required if x not in out]
    if missing:
        raise ValueError(f"{source_name}: missing {missing}; columns={list(df.columns)}")
    x = df.rename(columns={v:k for k,v in out.items()}).copy()
    if "symbol" not in x.columns:
        x["symbol"] = Path(source_name).stem
    x["Date"] = pd.to_datetime(x["date"], errors="coerce")
    for c in ["open","high","low","close","volume"]:
        x[c.title() if c != "volume" else "Volume"] = pd.to_numeric(x[c], errors="coerce")
    x = x.rename(columns={"open":"Open","high":"High","low":"Low","close":"Close"})
    if "Volume" not in x.columns:
        x["Volume"] = pd.to_numeric(x["volume"], errors="coerce")
    x["Symbol"] = x["symbol"].astype(str).str.upper().str.strip()
    x = x[["Date","Symbol","Open","High","Low","Close","Volume"]]
    return x

frames = []
validation = []
for f in stock_files:
    try:
        raw = pd.read_parquet(f)
        x = normalize_stock(raw, f.name)
        validation.append({
            "file": f.name,
            "rows": len(x),
            "min_date": x["Date"].min(),
            "max_date": x["Date"].max(),
            "symbols": x["Symbol"].nunique(),
            "bad_dates": int(x["Date"].isna().sum()),
        })
        frames.append(x)
    except Exception as e:
        validation.append({"file": f.name, "error": str(e)})

validation_df = pd.DataFrame(validation)
print(validation_df.head())
print("Files with errors:", validation_df["error"].notna().sum() if "error" in validation_df else 0)

stocks = pd.concat(frames, ignore_index=True)
stocks = stocks.drop_duplicates(["Symbol","Date"]).sort_values(["Symbol","Date"]).reset_index(drop=True)
print("Rows:", len(stocks), "Symbols:", stocks["Symbol"].nunique())
print("Date range:", stocks["Date"].min(), "to", stocks["Date"].max())

## 5. Validate stock data

In [ ]:
bad = {
    "null_date": int(stocks["Date"].isna().sum()),
    "null_ohlcv": int(stocks[["Open","High","Low","Close","Volume"]].isna().any(axis=1).sum()),
    "nonpositive_close": int((stocks["Close"] <= 0).sum()),
    "high_below_low": int((stocks["High"] < stocks["Low"]).sum()),
    "open_outside_extreme": int(((stocks["Open"] > stocks["High"]) | (stocks["Open"] < stocks["Low"])).sum()),
    "close_outside_extreme": int(((stocks["Close"] > stocks["High"]) | (stocks["Close"] < stocks["Low"])).sum()),
    "negative_volume": int((stocks["Volume"] < 0).sum()),
}
print(json.dumps(bad, indent=2))
assert bad["null_date"] == 0
assert bad["nonpositive_close"] == 0
assert bad["high_below_low"] == 0

## 6. Load and validate NIFTY50

In [ ]:
nifty_raw = pd.read_parquet(NIFTY_PATH)
print("NIFTY columns:", list(nifty_raw.columns))
nifty = nifty_raw.copy()
cols = {c.lower().strip(): c for c in nifty.columns}
rename = {}
for target, candidates in {
    "Date": ["date","datetime","timestamp"],
    "Open": ["open"], "High": ["high"], "Low": ["low"], "Close": ["close"]
}.items():
    found = next((cols[c] for c in candidates if c in cols), None)
    if found is None:
        raise ValueError(f"NIFTY missing {target}")
    rename[found] = target
nifty = nifty.rename(columns=rename)[["Date","Open","High","Low","Close"]]
nifty["Date"] = pd.to_datetime(nifty["Date"], errors="coerce")
for c in ["Open","High","Low","Close"]:
    nifty[c] = pd.to_numeric(nifty[c], errors="coerce")
nifty = nifty.dropna().drop_duplicates("Date").sort_values("Date").reset_index(drop=True)

print("NIFTY rows:", len(nifty))
print("NIFTY range:", nifty["Date"].min(), "to", nifty["Date"].max())
assert (nifty["Close"] > 0).all()
assert (nifty["High"] >= nifty["Low"]).all()

## 7. Build causal NIFTY regime

In [ ]:
n = nifty.copy()
n["SMA20"] = n["Close"].rolling(20).mean()
n["SMA50"] = n["Close"].rolling(50).mean()
n["SMA200"] = n["Close"].rolling(200).mean()
n["Return20"] = n["Close"].pct_change(20)
n["Return60"] = n["Close"].pct_change(60)
prev_close = n["Close"].shift(1)
tr = pd.concat([
    n["High"] - n["Low"],
    (n["High"] - prev_close).abs(),
    (n["Low"] - prev_close).abs()
], axis=1).max(axis=1)
n["ATR14"] = tr.rolling(14).mean()
n["ATRPct"] = n["ATR14"] / n["Close"]
n["SMA200Slope20"] = n["SMA200"] / n["SMA200"].shift(20) - 1
n["Above200"] = n["Close"] > n["SMA200"]
n["50Above200"] = n["SMA50"] > n["SMA200"]

enough = n["SMA200"].notna() & n["Return20"].notna()
n["Regime"] = "UNKNOWN"
n.loc[enough & n["Above200"] & n["50Above200"] & (n["Return20"] > 0), "Regime"] = "BULL"
n.loc[enough & (n["Close"] < n["SMA200"]) & (~n["50Above200"]) & (n["Return20"] < 0), "Regime"] = "BEAR"
n.loc[enough & (n["Regime"] == "UNKNOWN"), "Regime"] = "MIXED"

nifty_regime = n.rename(columns={
    "Close":"NiftyClose",
    "SMA20":"NiftySMA20",
    "SMA50":"NiftySMA50",
    "SMA200":"NiftySMA200",
    "Return20":"NiftyReturn20",
    "Return60":"NiftyReturn60",
    "ATR14":"NiftyATR14",
    "ATRPct":"NiftyATRPct",
    "SMA200Slope20":"NiftySMA200Slope20",
    "Above200":"NiftyAbove200",
    "50Above200":"Nifty50Above200",
    "Regime":"NiftyRegime"
})[["Date","NiftyClose","NiftySMA20","NiftySMA50","NiftySMA200","NiftyReturn20","NiftyReturn60","NiftyATR14","NiftyATRPct","NiftySMA200Slope20","NiftyAbove200","Nifty50Above200","NiftyRegime"]]

nifty_regime.to_parquet(NIFTY_REGIME_PATH, index=False)
print(nifty_regime["NiftyRegime"].value_counts(dropna=False))

## 8. Stock feature engineering

In [ ]:
df = stocks.copy()
g = df.groupby("Symbol", group_keys=False)

for nwin in [20, 50, 100, 200]:
    df[f"SMA{nwin}"] = g["Close"].transform(lambda s, nwin=nwin: s.rolling(nwin).mean())

for nwin in [5, 20, 60, 120, 252]:
    df[f"Return{nwin}"] = g["Close"].transform(lambda s, nwin=nwin: s.pct_change(nwin))

df["SMA50_to_SMA200"] = df["SMA50"] / df["SMA200"] - 1
df["SMA20_to_SMA50"] = df["SMA20"] / df["SMA50"] - 1
df["Close_to_SMA20"] = df["Close"] / df["SMA20"] - 1
df["Close_to_SMA50"] = df["Close"] / df["SMA50"] - 1
df["Close_to_SMA200"] = df["Close"] / df["SMA200"] - 1
df["SMA200Slope20"] = g["SMA200"].transform(lambda s: s / s.shift(20) - 1)

prev_close = g["Close"].shift(1)
df["TR"] = pd.concat([
    df["High"] - df["Low"],
    (df["High"] - prev_close).abs(),
    (df["Low"] - prev_close).abs()
], axis=1).max(axis=1)
for nwin in [14, 20, 50]:
    df[f"ATR{nwin}"] = g["TR"].transform(lambda s, nwin=nwin: s.rolling(nwin).mean())
df["ATRPct14"] = df["ATR14"] / df["Close"]
df["ATRPct20"] = df["ATR20"] / df["Close"]

for nwin in [20, 60]:
    df[f"VolumeSMA{nwin}"] = g["Volume"].transform(lambda s, nwin=nwin: s.rolling(nwin).mean())
df["VolumeRatio20"] = df["Volume"] / df["VolumeSMA20"]
df["VolumeRatio60"] = df["Volume"] / df["VolumeSMA60"]

for nwin in [20, 60, 120, 252]:
    prior_high = g["High"].transform(lambda s, nwin=nwin: s.rolling(nwin).max().shift(1))
    df[f"Prior{nwin}High"] = prior_high
    df[f"Breakout{nwin}"] = df["Close"] > prior_high
    df[f"Distance{nwin}High"] = df["Close"] / prior_high - 1

df["Volatility20"] = g["Close"].transform(lambda s: s.pct_change().rolling(20).std())
df["Volatility60"] = g["Close"].transform(lambda s: s.pct_change().rolling(60).std())
df["ATRCompression"] = df["ATRPct20"] / g["ATRPct20"].transform(lambda s: s.rolling(60).mean())

df = df.merge(nifty_regime, on="Date", how="left", validate="many_to_one")

df["RS20"] = df["Return20"] - df["NiftyReturn20"]
df["RS60"] = df["Return60"] - df["NiftyReturn60"]
df["RS120"] = df["Return120"] - df["NiftyReturn60"] * 2  # simple relative-strength proxy using available NIFTY return horizons

df["HistoryCount"] = g.cumcount() + 1
print("Feature rows:", len(df))

## 9. Daily cross-sectional ranks and composite score

In [ ]:
# Percentile ranks are computed separately on each date.
# Higher is better for momentum/RS/trend/breakout/volume.
# Lower ATR% is generally better for the volatility component.

rank_features = [
    "Return20", "Return60", "Return120",
    "RS20", "RS60", "RS120",
    "Close_to_SMA50", "Close_to_SMA200", "SMA200Slope20",
    "Distance60High", "Distance120High",
    "VolumeRatio20",
]

for c in rank_features:
    df[c + "_Rank"] = df.groupby("Date")[c].rank(pct=True, method="average")

df["Volatility_Rank"] = df.groupby("Date")["ATRPct20"].rank(pct=True, method="average")
df["VolatilityScore"] = 1 - df["Volatility_Rank"]

df["MomentumScore"] = df[["Return20_Rank","Return60_Rank","Return120_Rank"]].mean(axis=1)
df["RSScore"] = df[["RS20_Rank","RS60_Rank","RS120_Rank"]].mean(axis=1)
df["TrendScore"] = df[["Close_to_SMA50_Rank","Close_to_SMA200_Rank","SMA200Slope20_Rank"]].mean(axis=1)
df["BreakoutScore"] = df[["Distance60High_Rank","Distance120High_Rank"]].mean(axis=1)
df["VolumeScore"] = df["VolumeRatio20_Rank"]

w = CFG["weights"]
df["CompositeScore"] = (
    w["momentum"] * df["MomentumScore"] +
    w["relative_strength"] * df["RSScore"] +
    w["trend"] * df["TrendScore"] +
    w["breakout"] * df["BreakoutScore"] +
    w["volume"] * df["VolumeScore"] +
    w["volatility"] * df["VolatilityScore"]
)

df["DailyRank"] = df.groupby("Date")["CompositeScore"].rank(method="first", ascending=False)
df["UniverseCount"] = df.groupby("Date")["CompositeScore"].transform("count")
df["RankPct"] = df["DailyRank"] / df["UniverseCount"]

df["Eligible"] = (
    (df["HistoryCount"] >= CFG["min_history"]) &
    (df["Close"] >= CFG["min_price"]) &
    (df["Volume"] > 0) &
    df["CompositeScore"].notna() &
    df["NiftyRegime"].isin(["BULL","MIXED","BEAR"])
)

df["EntryCandidate"] = df["Eligible"] & (df["RankPct"] <= CFG["entry_top_pct"])
if CFG["require_breakout"]:
    df["EntryCandidate"] &= (df["Breakout60"] | df["Breakout120"])

print(df["EntryCandidate"].mean())

## 10. Save feature dataset

In [ ]:
feature_cols = [
    "Date","Symbol","Open","High","Low","Close","Volume",
    "SMA20","SMA50","SMA100","SMA200",
    "Return5","Return20","Return60","Return120","Return252",
    "SMA50_to_SMA200","SMA20_to_SMA50","Close_to_SMA20","Close_to_SMA50","Close_to_SMA200","SMA200Slope20",
    "TR","ATR14","ATR20","ATR50","ATRPct14","ATRPct20",
    "VolumeSMA20","VolumeSMA60","VolumeRatio20","VolumeRatio60",
    "Breakout20","Breakout60","Breakout120","Breakout252",
    "Distance20High","Distance60High","Distance120High","Distance252High",
    "Volatility20","Volatility60","ATRCompression",
    "NiftyClose","NiftySMA20","NiftySMA50","NiftySMA200","NiftyReturn20","NiftyReturn60",
    "NiftyATR14","NiftyATRPct","NiftySMA200Slope20","NiftyAbove200","Nifty50Above200","NiftyRegime",
    "RS20","RS60","RS120",
    "MomentumScore","RSScore","TrendScore","BreakoutScore","VolumeScore","VolatilityScore",
    "CompositeScore","DailyRank","UniverseCount","RankPct","Eligible","EntryCandidate"
]
feature_cols = [c for c in feature_cols if c in df.columns]
features = df[feature_cols].copy()
features.to_parquet(STOCK_FEATURES_PATH, index=False)
print("Saved:", STOCK_FEATURES_PATH, features.shape)

## 11. Build a causal trade simulator

The simulator operates stock-by-stock. It generates an entry only after a close-T signal and executes at T+1 Open. Exit signals are similarly generated at close-T and executed at T+1 Open.

In [ ]:
def round_trip_cost_bps(cfg):
    return (
        cfg["brokerage_bps"] +
        cfg["stt_bps"] +
        cfg["exchange_bps"] +
        cfg["stamp_bps"] +
        cfg["sebi_bps"]
    )

def execution_price(raw_price, side, slippage_bps):
    if side == "BUY":
        return raw_price * (1 + slippage_bps / 10000)
    return raw_price * (1 - slippage_bps / 10000)

def transaction_cost(notional, cfg, side):
    brokerage = notional * cfg["brokerage_bps"] / 10000
    stt = notional * cfg["stt_bps"] / 10000 if side == "SELL" else 0.0
    exchange = notional * cfg["exchange_bps"] / 10000
    stamp = notional * cfg["stamp_bps"] / 10000 if side == "BUY" else 0.0
    sebi = notional * cfg["sebi_bps"] / 10000
    gst = (brokerage + exchange + sebi) * cfg["gst_rate"]
    return brokerage + stt + exchange + stamp + sebi + gst

def prepare_symbol_frames(features):
    frames = {}
    for sym, x in features.groupby("Symbol", sort=False):
        x = x.sort_values("Date").reset_index(drop=True).copy()
        x["NextOpen"] = x["Open"].shift(-1)
        x["NextDate"] = x["Date"].shift(-1)
        x["NextVolume"] = x["Volume"].shift(-1)
        x["NextClose"] = x["Close"].shift(-1)
        frames[sym] = x
    return frames

## 12. Single-stock causal trade generation

In [ ]:
def simulate_symbol_trades(x, cfg):
    x = x.sort_values("Date").reset_index(drop=True)
    trades = []
    in_pos = False
    entry_idx = None
    entry_date = None
    entry_price = None
    shares = None
    highest_close = None
    entry_cost = 0.0
    entry_nifty_regime = None
    entry_score = None

    for i in range(len(x) - 1):
        row = x.iloc[i]
        nxt = x.iloc[i + 1]

        if pd.isna(nxt["Open"]) or nxt["Open"] <= 0:
            continue

        # Entry is decided at close T and executed at open T+1.
        if not in_pos:
            if not bool(row.get("EntryCandidate", False)):
                continue

            regime = row["NiftyRegime"]
            exposure = cfg["regime_exposure"].get(regime, 0.0)
            if exposure <= 0:
                continue

            raw_entry = float(nxt["Open"])
            entry_price = execution_price(raw_entry, "BUY", cfg["slippage_bps"])

            # Unit notional is used here; portfolio-level sizing happens later.
            in_pos = True
            entry_idx = i + 1
            entry_date = nxt["Date"]
            shares = 1.0
            highest_close = float(row["Close"])
            entry_cost = transaction_cost(entry_price * shares, cfg, "BUY")
            entry_nifty_regime = regime
            entry_score = row["CompositeScore"]
            continue

        highest_close = max(highest_close, float(row["Close"]))

        holding_days = i - entry_idx + 1
        atr = row["ATR14"]
        atr_stop = highest_close - cfg["atr_trail_multiple"] * atr if pd.notna(atr) else -np.inf

        trend_exit = pd.notna(row["SMA50"]) and row["Close"] < row["SMA50"]
        momentum_exit = pd.notna(row["RankPct"]) and row["RankPct"] > (1 - cfg["momentum_exit_percentile"])
        time_exit = holding_days >= cfg["time_stop_days"]
        atr_exit = pd.notna(atr) and row["Close"] < atr_stop

        exit_reason = None
        if cfg["exit_mode"] == "ATR_TRAIL" and atr_exit:
            exit_reason = "ATR_TRAIL"
        elif trend_exit:
            exit_reason = "TREND"
        elif momentum_exit:
            exit_reason = "RANK"
        elif time_exit:
            exit_reason = "TIME"

        if exit_reason is not None:
            raw_exit = float(nxt["Open"])
            exit_price = execution_price(raw_exit, "SELL", cfg["slippage_bps"])
            gross_pnl = (exit_price - entry_price) * shares
            exit_cost = transaction_cost(exit_price * shares, cfg, "SELL")
            net_pnl = gross_pnl - entry_cost - exit_cost
            net_return = net_pnl / max(entry_price * shares, 1e-12)
            trades.append({
                "Symbol": x["Symbol"].iloc[0],
                "SignalDate": row["Date"],
                "EntryDate": entry_date,
                "EntryPrice": entry_price,
                "ExitSignalDate": row["Date"],
                "ExitDate": nxt["Date"],
                "ExitPrice": exit_price,
                "GrossPnL": gross_pnl,
                "EntryCost": entry_cost,
                "ExitCost": exit_cost,
                "NetPnL": net_pnl,
                "NetReturn": net_return,
                "HoldingDays": holding_days,
                "ExitReason": exit_reason,
                "NiftyRegime": entry_nifty_regime,
                "EntryScore": entry_score,
                "ATRAtExit": atr,
                "Turnover": (entry_price + exit_price) * shares,
            })
            in_pos = False
            entry_idx = None
            entry_date = None
            entry_price = None
            shares = None
            highest_close = None
            entry_cost = 0.0
            entry_nifty_regime = None
            entry_score = None

    return pd.DataFrame(trades)

## 13. Generate baseline causal trade candidates

In [ ]:
symbol_frames = prepare_symbol_frames(features)
all_trades = []
for sym, x in symbol_frames.items():
    try:
        t = simulate_symbol_trades(x, CFG)
        if not t.empty:
            all_trades.append(t)
    except Exception as e:
        print("Simulation error", sym, e)

trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
print("Closed trades:", len(trades))
if not trades.empty:
    print(trade_metrics(trades))
    print(trades.head())

## 14. Portfolio-level ranking, sizing and capacity model

The stock-level trade list above is useful for diagnostics. The portfolio engine below constructs the actual daily portfolio from the ranked candidates, applies NIFTY exposure, volatility sizing, max position weight, and liquidity participation.

In [ ]:
def build_portfolio(features, cfg):
    x = features.sort_values(["Date","Symbol"]).copy()
    dates = sorted(x["Date"].dropna().unique())

    cash = cfg["initial_capital"]
    positions = {}  # symbol -> dict
    equity_rows = []
    trade_rows = []

    grouped = {d: g.copy() for d, g in x.groupby("Date", sort=False)}

    for d in dates[:-1]:
        day = grouped[d]
        next_date = dates[dates.index(d)+1]
        next_day = grouped[next_date].set_index("Symbol")

        # 1) Generate exits at close T; execute at next open.
        for sym in list(positions.keys()):
            if sym not in day["Symbol"].values or sym not in next_day.index:
                continue
            r = day[day["Symbol"] == sym].iloc[0]
            pos = positions[sym]

            pos["highest_close"] = max(pos["highest_close"], float(r["Close"]))
            atr = r["ATR14"]
            atr_stop = pos["highest_close"] - cfg["atr_trail_multiple"] * atr if pd.notna(atr) else -np.inf
            trend_exit = pd.notna(r["SMA50"]) and r["Close"] < r["SMA50"]
            rank_exit = pd.notna(r["RankPct"]) and r["RankPct"] > (1 - cfg["momentum_exit_percentile"])
            time_exit = (d - pos["entry_signal_date"]).days >= cfg["time_stop_days"]
            atr_exit = pd.notna(atr) and r["Close"] < atr_stop

            reason = None
            if cfg["exit_mode"] == "ATR_TRAIL" and atr_exit:
                reason = "ATR_TRAIL"
            elif trend_exit:
                reason = "TREND"
            elif rank_exit:
                reason = "RANK"
            elif time_exit:
                reason = "TIME"

            if reason:
                raw = float(next_day.loc[sym, "Open"])
                px = execution_price(raw, "SELL", cfg["slippage_bps"])
                notional = pos["shares"] * px
                cost = transaction_cost(notional, cfg, "SELL")
                proceeds = notional - cost
                cash += proceeds
                gross = (px - pos["entry_price"]) * pos["shares"]
                net = gross - pos["entry_cost"] - cost

                trade_rows.append({
                    "Symbol": sym,
                    "SignalDate": pos["entry_signal_date"],
                    "EntryDate": pos["entry_date"],
                    "EntryPrice": pos["entry_price"],
                    "ExitSignalDate": d,
                    "ExitDate": next_date,
                    "ExitPrice": px,
                    "Shares": pos["shares"],
                    "GrossPnL": gross,
                    "EntryCost": pos["entry_cost"],
                    "ExitCost": cost,
                    "NetPnL": net,
                    "NetReturn": net / max(pos["entry_notional"], 1e-12),
                    "HoldingDays": (next_date - pos["entry_date"]).days,
                    "ExitReason": reason,
                    "NiftyRegime": pos["regime"],
                    "EntryScore": pos["score"],
                    "Turnover": pos["entry_notional"] + notional,
                })
                del positions[sym]

        # 2) Mark current portfolio.
        marked = cash
        for sym, pos in positions.items():
            if sym in day["Symbol"].values:
                px = float(day.loc[day["Symbol"] == sym, "Close"].iloc[0])
                marked += pos["shares"] * px

        # 3) New entries from close-T ranking, executed at next open.
        candidates = day[
            day["EntryCandidate"] &
            day["NiftyRegime"].map(cfg["regime_exposure"]).fillna(0).gt(0)
        ].copy()
        candidates = candidates.sort_values("CompositeScore", ascending=False).head(cfg["max_positions"])

        current_value = marked
        regime = day["NiftyRegime"].dropna().iloc[0] if len(day["NiftyRegime"].dropna()) else "UNKNOWN"
        target_exposure = cfg["regime_exposure"].get(regime, 0.0) * cfg["max_total_exposure"]

        for _, r in candidates.iterrows():
            sym = r["Symbol"]
            if sym in positions or sym not in next_day.index:
                continue
            if len(positions) >= cfg["max_positions"]:
                break

            raw_open = float(next_day.loc[sym, "Open"])
            if not np.isfinite(raw_open) or raw_open <= 0:
                continue

            atr = float(r["ATR14"]) if pd.notna(r["ATR14"]) else np.nan
            if not np.isfinite(atr) or atr <= 0:
                continue

            avg_tv = float(r["VolumeSMA20"] * r["Close"]) if pd.notna(r["VolumeSMA20"]) else np.nan
            if not np.isfinite(avg_tv) or avg_tv <= 0:
                continue

            target_value = current_value * min(
                cfg["max_position_weight"],
                target_exposure / max(cfg["max_positions"], 1)
            )

            # Volatility sizing: approximately 1% portfolio risk to a k*ATR stop.
            risk_per_share = cfg["atr_stop_multiple"] * atr
            vol_value = current_value * cfg["risk_budget_per_position"] / risk_per_share * raw_open
            target_value = min(target_value, vol_value)

            # Liquidity capacity.
            target_value = min(target_value, avg_tv * cfg["max_participation_pct"])
            target_value = min(target_value, cash)

            if target_value <= 0:
                continue

            px = execution_price(raw_open, "BUY", cfg["slippage_bps"])
            shares = math.floor(target_value / px)
            if shares <= 0:
                continue

            notional = shares * px
            cost = transaction_cost(notional, cfg, "BUY")
            total_cash = notional + cost
            if total_cash > cash:
                shares = math.floor(cash / max(px * 1.001, 1e-12))
                if shares <= 0:
                    continue
                notional = shares * px
                cost = transaction_cost(notional, cfg, "BUY")
                total_cash = notional + cost

            if total_cash > cash:
                continue

            cash -= total_cash
            positions[sym] = {
                "shares": shares,
                "entry_price": px,
                "entry_notional": notional,
                "entry_cost": cost,
                "entry_signal_date": d,
                "entry_date": next_date,
                "highest_close": float(r["Close"]),
                "regime": r["NiftyRegime"],
                "score": float(r["CompositeScore"]),
            }

        gross_exposure = 0.0
        for sym, pos in positions.items():
            if sym in day["Symbol"].values:
                px = float(day.loc[day["Symbol"] == sym, "Close"].iloc[0])
                gross_exposure += pos["shares"] * px

        equity_rows.append({
            "Date": d,
            "Cash": cash,
            "GrossExposure": gross_exposure,
            "PortfolioValue": cash + gross_exposure,
            "Positions": len(positions),
            "NiftyRegime": regime,
        })

    equity = pd.DataFrame(equity_rows)
    trades = pd.DataFrame(trade_rows)
    return equity, trades

equity, portfolio_trades = build_portfolio(features, CFG)
print("Equity rows:", len(equity))
print("Closed portfolio trades:", len(portfolio_trades))

## 15. Save portfolio and trades

In [ ]:
equity.to_parquet(EQUITY_PATH, index=False)
equity.to_csv(RESULTS_DIR / "portfolio" / "equity_curve.csv", index=False)

portfolio_trades.to_parquet(TRADES_PATH, index=False)
portfolio_trades.to_csv(RESULTS_DIR / "trades" / "trades.csv", index=False)

print("Saved portfolio:", EQUITY_PATH)
print("Saved trades:", TRADES_PATH)

## 16. Performance report

In [ ]:
perf = performance_metrics(equity)
tm = trade_metrics(portfolio_trades)
summary = {**perf, **tm}
summary_df = pd.DataFrame([summary])
display(summary_df.T)

## 17. Equity curve and drawdown

In [ ]:
e = equity.copy()
e["Date"] = pd.to_datetime(e["Date"])
e = e.sort_values("Date")
e["Peak"] = e["PortfolioValue"].cummax()
e["Drawdown"] = e["PortfolioValue"] / e["Peak"] - 1

fig, ax = plt.subplots(figsize=(14,5))
ax.plot(e["Date"], e["PortfolioValue"])
ax.set_title("V5 Portfolio Equity Curve")
ax.set_ylabel("Portfolio Value")
ax.grid(True, alpha=0.2)
plt.show()

fig, ax = plt.subplots(figsize=(14,4))
ax.plot(e["Date"], e["Drawdown"])
ax.set_title("V5 Drawdown")
ax.set_ylabel("Drawdown")
ax.grid(True, alpha=0.2)
plt.show()

## 18. Regime attribution

In [ ]:
if not portfolio_trades.empty:
    regime_stats = portfolio_trades.groupby("NiftyRegime").agg(
        Trades=("NetReturn","size"),
        AvgReturn=("NetReturn","mean"),
        MedianReturn=("NetReturn","median"),
        WinRate=("NetReturn", lambda s: (s>0).mean()),
        TotalPnL=("NetPnL","sum"),
        AvgHolding=("HoldingDays","mean"),
    ).reset_index()
else:
    regime_stats = pd.DataFrame()
display(regime_stats)
regime_stats.to_csv(RESULTS_DIR / "reports" / "regime_analysis.csv", index=False)

## 19. Yearly returns

In [ ]:
e["Year"] = e["Date"].dt.year
yearly = []
for year, x in e.groupby("Year"):
    yearly.append({
        "Year": year,
        "Return": x["PortfolioValue"].iloc[-1] / x["PortfolioValue"].iloc[0] - 1,
        "MaxDrawdown": max_drawdown(x.set_index("Date")["PortfolioValue"]),
        "AvgPositions": x["Positions"].mean(),
    })
yearly_df = pd.DataFrame(yearly)
display(yearly_df)
yearly_df.to_csv(RESULTS_DIR / "reports" / "yearly_returns.csv", index=False)

## 20. Walk-forward framework

The walk-forward implementation uses chronological annual test windows. Parameters are frozen from configuration rather than optimized using future test data. This provides an honest baseline OOS framework. A later research iteration can add parameter selection strictly inside each training window.

In [ ]:
def date_years(df):
    return sorted(pd.to_datetime(df["Date"]).dt.year.dropna().unique())

def run_period_backtest(features, cfg, start_date=None, end_date=None):
    x = features.copy()
    if start_date is not None:
        x = x[x["Date"] >= pd.Timestamp(start_date)]
    if end_date is not None:
        x = x[x["Date"] <= pd.Timestamp(end_date)]
    if x.empty:
        return pd.DataFrame(), pd.DataFrame()
    return build_portfolio(x, cfg)

years = date_years(features)
wf_rows = []

# Expanding-window walk-forward:
# Train history is descriptive; parameters are fixed from CFG.
# Each test year is evaluated separately and is never used to set parameters.
for test_year in years:
    if test_year < years[0] + 3:
        continue
    test_start = f"{test_year}-01-01"
    test_end = f"{test_year}-12-31"
    eq_y, tr_y = run_period_backtest(features, CFG, test_start, test_end)
    if len(eq_y) < 20:
        continue
    pm = performance_metrics(eq_y)
    tm_y = trade_metrics(tr_y)
    wf_rows.append({
        "TestYear": test_year,
        "CAGR": pm.get("CAGR"),
        "TotalReturn": pm.get("TotalReturn"),
        "MaxDrawdown": pm.get("MaxDrawdown"),
        "Sharpe": pm.get("Sharpe"),
        "Sortino": pm.get("Sortino"),
        "Trades": tm_y.get("Trades", 0),
        "WinRate": tm_y.get("WinRate", np.nan),
        "ProfitFactor": tm_y.get("ProfitFactor", np.nan),
    })

wf = pd.DataFrame(wf_rows)
display(wf)
wf.to_csv(WF_RESULTS_PATH, index=False)

## 21. Walk-forward aggregate

In [ ]:
if not wf.empty:
    print("Walk-forward median CAGR:", wf["CAGR"].median())
    print("Walk-forward median MaxDD:", wf["MaxDrawdown"].median())
    print("Positive-return years:", (wf["TotalReturn"] > 0).mean())
    print("Years:", len(wf))

## 22. Monte Carlo analysis

Monte Carlo resamples the realized closed-trade return distribution with replacement. It is a robustness analysis of trade-sequence uncertainty, not a forecast of future market returns.

In [ ]:
def monte_carlo_trade_paths(trades, initial_capital, n_sims=10000, seed=42):
    if trades.empty:
        return pd.DataFrame()

    rng = np.random.default_rng(seed)
    r = trades["NetReturn"].replace([np.inf,-np.inf], np.nan).dropna().values
    r = r[np.isfinite(r)]
    if len(r) < 5:
        return pd.DataFrame()

    out = np.empty((n_sims, 5))
    n = len(r)

    for i in range(n_sims):
        sample = rng.choice(r, size=n, replace=True)
        wealth = initial_capital * np.cumprod(1 + sample)
        peak = np.maximum.accumulate(wealth)
        dd = wealth / peak - 1
        years = max(n / 252, 1/252)
        cagr = (wealth[-1] / initial_capital) ** (1 / years) - 1
        out[i] = [
            wealth[-1],
            cagr,
            dd.min(),
            np.mean(sample > 0),
            np.median(sample),
        ]

    return pd.DataFrame(out, columns=[
        "FinalCapital","CAGR","MaxDrawdown","WinRate","MedianTradeReturn"
    ])

mc = monte_carlo_trade_paths(
    portfolio_trades,
    CFG["initial_capital"],
    CFG["mc_simulations"],
    CFG["random_seed"]
)
print("Simulations:", len(mc))

## 23. Monte Carlo summary

In [ ]:
if not mc.empty:
    mc_summary = pd.DataFrame({
        "Metric": ["FinalCapital","CAGR","MaxDrawdown","WinRate","MedianTradeReturn"],
        "P05": [mc[c].quantile(.05) for c in ["FinalCapital","CAGR","MaxDrawdown","WinRate","MedianTradeReturn"]],
        "P25": [mc[c].quantile(.25) for c in ["FinalCapital","CAGR","MaxDrawdown","WinRate","MedianTradeReturn"]],
        "Median": [mc[c].median() for c in ["FinalCapital","CAGR","MaxDrawdown","WinRate","MedianTradeReturn"]],
        "P75": [mc[c].quantile(.75) for c in ["FinalCapital","CAGR","MaxDrawdown","WinRate","MedianTradeReturn"]],
        "P95": [mc[c].quantile(.95) for c in ["FinalCapital","CAGR","MaxDrawdown","WinRate","MedianTradeReturn"]],
    })
    display(mc_summary)
    mc_summary.to_csv(RESULTS_DIR / "reports" / "monte_carlo_summary.csv", index=False)
    mc.to_parquet(RESULTS_DIR / "monte_carlo" / "simulations.parquet", index=False)

## 24. Monte Carlo 2× / 5× analysis

In [ ]:
if not mc.empty:
    print("P(Final capital >= 2x):", (mc["FinalCapital"] >= 2*CFG["initial_capital"]).mean())
    print("P(Final capital >= 5x):", (mc["FinalCapital"] >= 5*CFG["initial_capital"]).mean())
    print("P(Max DD <= -50%):", (mc["MaxDrawdown"] <= -0.50).mean())

## 25. Best-trade dependency test

In [ ]:
def bootstrap_final_wealth(returns, initial_capital, n=10000, seed=42):
    rng = np.random.default_rng(seed)
    r = np.asarray(returns)
    if len(r) == 0:
        return np.array([])
    idx = rng.integers(0, len(r), size=(n, len(r)))
    samples = r[idx]
    return initial_capital * np.prod(1 + samples, axis=1)

if not portfolio_trades.empty:
    r = portfolio_trades["NetReturn"].sort_values(ascending=False).reset_index(drop=True)
    tests = []
    for remove_n in [0,1,5,10,max(1,int(len(r)*0.05))]:
        rr = r.iloc[remove_n:].values
        wealth = CFG["initial_capital"] * np.prod(1 + rr) if len(rr) else CFG["initial_capital"]
        tests.append({"RemovedBestTrades": remove_n, "FinalCapital": wealth})
    best_trade_test = pd.DataFrame(tests)
    display(best_trade_test)
    best_trade_test.to_csv(RESULTS_DIR / "reports" / "best_trade_dependency.csv", index=False)

## 26. Cost sensitivity

In [ ]:
cost_rows = []
for slip in [0, 10, 25, 50, 100, 150]:
    c = dict(CFG)
    c["slippage_bps"] = slip
    eq_c, tr_c = build_portfolio(features, c)
    pm = performance_metrics(eq_c) if len(eq_c) else {}
    tm_c = trade_metrics(tr_c)
    cost_rows.append({
        "SlippageBps": slip,
        "CAGR": pm.get("CAGR"),
        "MaxDrawdown": pm.get("MaxDrawdown"),
        "Sharpe": pm.get("Sharpe"),
        "Trades": tm_c.get("Trades",0),
        "ProfitFactor": tm_c.get("ProfitFactor",np.nan),
    })
cost_df = pd.DataFrame(cost_rows)
display(cost_df)
cost_df.to_csv(RESULTS_DIR / "reports" / "cost_sensitivity.csv", index=False)

## 27. Position-count sensitivity

In [ ]:
rows = []
for npos in [5, 10, 15, 20]:
    c = dict(CFG)
    c["max_positions"] = npos
    eq_c, tr_c = build_portfolio(features, c)
    pm = performance_metrics(eq_c) if len(eq_c) else {}
    tm_c = trade_metrics(tr_c)
    rows.append({
        "MaxPositions": npos,
        "CAGR": pm.get("CAGR"),
        "MaxDrawdown": pm.get("MaxDrawdown"),
        "Sharpe": pm.get("Sharpe"),
        "Trades": tm_c.get("Trades",0),
        "ProfitFactor": tm_c.get("ProfitFactor",np.nan),
    })
position_df = pd.DataFrame(rows)
display(position_df)
position_df.to_csv(RESULTS_DIR / "reports" / "position_count_sensitivity.csv", index=False)

## 28. Exit sensitivity

In [ ]:
rows = []
for k in [3,4,5,6]:
    c = dict(CFG)
    c["atr_trail_multiple"] = k
    eq_c, tr_c = build_portfolio(features, c)
    pm = performance_metrics(eq_c) if len(eq_c) else {}
    tm_c = trade_metrics(tr_c)
    rows.append({
        "ATRMultiple": k,
        "CAGR": pm.get("CAGR"),
        "MaxDrawdown": pm.get("MaxDrawdown"),
        "Sharpe": pm.get("Sharpe"),
        "Trades": tm_c.get("Trades",0),
        "ProfitFactor": tm_c.get("ProfitFactor",np.nan),
    })
exit_df = pd.DataFrame(rows)
display(exit_df)
exit_df.to_csv(RESULTS_DIR / "reports" / "exit_sensitivity.csv", index=False)

## 29. Liquidity / capacity diagnostics

In [ ]:
liq = features.copy()
liq["AvgTradedValue20"] = liq["VolumeSMA20"] * liq["Close"]
liq_stats = liq[["AvgTradedValue20"]].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99])
display(liq_stats)

if not portfolio_trades.empty:
    entry_symbols = portfolio_trades["Symbol"].value_counts().head(20)
    display(entry_symbols.to_frame("ClosedTrades"))

## 30. Strategy sanity checks

In [ ]:
assert features["Date"].notna().all()
assert features[["Open","High","Low","Close"]].notna().all().all()
assert (features["Close"] > 0).all()
assert (features["High"] >= features["Low"]).all()

if not portfolio_trades.empty:
    assert (pd.to_datetime(portfolio_trades["ExitDate"]) > pd.to_datetime(portfolio_trades["EntryDate"])).all()
    assert (pd.to_datetime(portfolio_trades["EntryDate"]) > pd.to_datetime(portfolio_trades["SignalDate"])).all()

print("Python sanity checks passed.")

## 31. DuckDB sanity checks

In [ ]:
con = duckdb.connect()

con.register("features_view", features)
con.register("nifty_view", nifty_regime)
con.register("trades_view", portfolio_trades if not portfolio_trades.empty else pd.DataFrame(columns=["Symbol","SignalDate","EntryDate","ExitDate","NetReturn"]))
con.register("equity_view", equity)

queries = {
    "feature_rows": "SELECT COUNT(*) AS n FROM features_view",
    "feature_duplicates": "SELECT COUNT(*) AS n FROM (SELECT Symbol, Date, COUNT(*) c FROM features_view GROUP BY 1,2 HAVING c > 1)",
    "feature_date_range": "SELECT MIN(Date) AS min_date, MAX(Date) AS max_date FROM features_view",
    "nifty_date_range": "SELECT MIN(Date) AS min_date, MAX(Date) AS max_date FROM nifty_view",
    "trade_count": "SELECT COUNT(*) AS n FROM trades_view",
    "bad_trade_order": "SELECT COUNT(*) AS n FROM trades_view WHERE EntryDate <= SignalDate OR ExitDate <= EntryDate",
    "equity_rows": "SELECT COUNT(*) AS n FROM equity_view",
}
for name, q in queries.items():
    print("\n", name)
    display(con.execute(q).df())

## 32. Final summary export

In [ ]:
final_summary = {
    **summary,
    "WalkForwardYears": int(len(wf)),
    "WalkForwardMedianCAGR": float(wf["CAGR"].median()) if not wf.empty else np.nan,
    "WalkForwardPositiveYearPct": float((wf["TotalReturn"] > 0).mean()) if not wf.empty else np.nan,
    "MonteCarloP5FinalCapital": float(mc["FinalCapital"].quantile(.05)) if not mc.empty else np.nan,
    "MonteCarloMedianFinalCapital": float(mc["FinalCapital"].median()) if not mc.empty else np.nan,
    "MonteCarloP95FinalCapital": float(mc["FinalCapital"].quantile(.95)) if not mc.empty else np.nan,
    "MonteCarloP5CAGR": float(mc["CAGR"].quantile(.05)) if not mc.empty else np.nan,
    "MonteCarloMedianCAGR": float(mc["CAGR"].median()) if not mc.empty else np.nan,
    "MonteCarloP95CAGR": float(mc["CAGR"].quantile(.95)) if not mc.empty else np.nan,
    "MonteCarloP5MaxDrawdown": float(mc["MaxDrawdown"].quantile(.05)) if not mc.empty else np.nan,
    "MonteCarloMedianMaxDrawdown": float(mc["MaxDrawdown"].median()) if not mc.empty else np.nan,
}

final_df = pd.DataFrame([final_summary])
final_df.to_csv(RESULTS_DIR / "reports" / "summary.csv", index=False)

manifest = {
    "strategy": "NSE 5x Turnaround V5",
    "data_sources": {
        "stocks": str(DATA_DIR),
        "nifty": str(NIFTY_PATH),
    },
    "constraints": [
        "OHLCV stock data only",
        "NIFTY OHLC only",
        "No external fundamental/flow/sector data",
        "Signal at close T, execution at open T+1",
    ],
    "config": CFG,
    "feature_path": str(STOCK_FEATURES_PATH),
    "nifty_regime_path": str(NIFTY_REGIME_PATH),
    "trades_path": str(TRADES_PATH),
    "equity_path": str(EQUITY_PATH),
}
with open(RESULTS_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

display(final_df.T)
print("All artifacts saved under:", RESULTS_DIR)

## 33. Interpretation rules

Use these rules when evaluating V5:

1. **Out-of-sample results matter more than full-sample results.**
2. A high CAGR with extreme drawdown is not automatically preferable.
3. If small changes in ATR multiple, position count, costs, or ranking weights cause large performance changes, the strategy may be fragile.
4. If transaction costs destroy returns, the strategy is not economically robust.
5. If liquidity constraints eliminate many trades, the unconstrained backtest is not representative.
6. If removing a handful of exceptional trades destroys the result, the strategy is highly dependent on rare winners.
7. Monte Carlo describes trade-sequence uncertainty; it is not a guarantee of future performance.
8. V5 should remain a research system until it survives independent paper trading and live execution checks.

## 34. Important implementation note

This notebook deliberately does **not** use external data. It also does not pretend that missing historical information exists.

The next research iteration can improve the walk-forward parameter-selection process, add more rigorous event-level execution simulation, and compare V4 vs V5 under identical dates/cost assumptions—but only using the same available Parquet datasets.